1. Install yfinance library using pip

2. Create a function get_price_lastweek(ticker) that get the price of a US stock ticker for last week (Monday to Friday)

3. Call the function to get the prices of AAPL, MSFT, NVDA, TSLA, MS, GS for  last week .

4. Find which one has the highest return (i.e. Friday close price over Monday open price) during last week.

In [ ]:
!pip install yfinance

In [5]:
import yfinance as yf
from datetime import datetime, timedelta
import pandas as pd


In [6]:
def get_price_lastweek(ticker):
    """
    Fetches the daily OHLCV data for a given ticker
    for the most recent completed Mon–Fri week.
    """
    today = datetime.today()

    # Find last Monday and last Friday
    # weekday(): Monday=0, Sunday=6
    days_since_monday = today.weekday() + 7   # go back to LAST week's Monday
    last_monday = today - timedelta(days=days_since_monday)
    last_friday = last_monday + timedelta(days=4)

    start_date = last_monday.strftime("%Y-%m-%d")
    end_date   = (last_friday + timedelta(days=1)).strftime("%Y-%m-%d")  # yfinance end is exclusive

    stock = yf.Ticker(ticker)
    df = stock.history(start=start_date, end=end_date, interval="1d")

    if df.empty:
        print(f"⚠️  No data found for {ticker}")
        return None

    df.index = df.index.strftime("%Y-%m-%d")   # clean up the index display
    print(f"\n📈 {ticker} — {start_date} to {last_friday.strftime('%Y-%m-%d')}")
    print(df[["Open", "High", "Low", "Close", "Volume"]].to_string())

    return df

In [7]:
tickers = ["AAPL", "MSFT", "NVDA", "TSLA", "MS", "GS"]
data    = {}

for ticker in tickers:
    df = get_price_lastweek(ticker)
    if df is not None:
        data[ticker] = df


📈 AAPL — 2026-02-16 to 2026-02-20
                  Open        High         Low       Close    Volume
Date                                                                
2026-02-17  258.049988  266.290009  255.539993  263.880005  58469100
2026-02-18  263.600006  266.820007  262.450012  264.350006  34203300
2026-02-19  262.600006  264.480011  260.049988  260.579987  30845300
2026-02-20  258.970001  264.750000  258.160004  264.579987  42070500

📈 MSFT — 2026-02-16 to 2026-02-20
                  Open        High         Low       Close    Volume
Date                                                                
2026-02-17  398.310879  399.607907  393.631557  395.956238  32078800
2026-02-18  397.223349  401.643253  395.417473  398.690002  23223400
2026-02-19  400.690002  404.429993  396.670013  398.459991  28234000
2026-02-20  396.109985  400.119995  395.160004  397.230011  34015200

📈 NVDA — 2026-02-16 to 2026-02-20
                  Open        High         Low       Close     Volu

In [8]:
print("\n" + "="*55)
print("  Weekly Return Summary (Friday Close / Monday Open - 1)")
print("="*55)

returns = {}

for ticker, df in data.items():
    monday_open  = df["Open"].iloc[0]    # first day = Monday
    friday_close = df["Close"].iloc[-1]  # last  day = Friday

    weekly_return = (friday_close - monday_open) / monday_open * 100   # in %
    returns[ticker] = weekly_return

    print(f"  {ticker:<6}  Monday Open: {monday_open:>9.2f}  |  "
          f"Friday Close: {friday_close:>9.2f}  |  Return: {weekly_return:>+7.2f}%")

# Find the winner
best_ticker = max(returns, key=returns.get)

print("="*55)
print(f"\n🏆 Best performer last week: {best_ticker}")
print(f"   Weekly Return: {returns[best_ticker]:+.2f}%")


  Weekly Return Summary (Friday Close / Monday Open - 1)
  AAPL    Monday Open:    258.05  |  Friday Close:    264.58  |  Return:   +2.53%
  MSFT    Monday Open:    398.31  |  Friday Close:    397.23  |  Return:   -0.27%
  NVDA    Monday Open:    181.75  |  Friday Close:    189.82  |  Return:   +4.44%
  TSLA    Monday Open:    412.36  |  Friday Close:    411.82  |  Return:   -0.13%
  MS      Monday Open:    172.55  |  Friday Close:    175.41  |  Return:   +1.66%
  GS      Monday Open:    907.73  |  Friday Close:    922.24  |  Return:   +1.60%

🏆 Best performer last week: NVDA
   Weekly Return: +4.44%
